# Exemplo: Regressão multioutput
---------------------------------

Este exemplo mostra como usar o ExperionML para fazer predições em um conjunto de dados de regressão multioutput. Um dos modelos usados é um regressor MLP implementado com [Keras](https://keras.io/) usando [scikeras](https://www.adriangb.com/scikeras/refs/heads/master/index.html).

Os dados utilizados são um conjunto sintético criado com a função [make_regression](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_regression.html) do sklearn.

## Carregar os dados

In [1]:
# Disable annoying tf warnings
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from tensorflow import get_logger
get_logger().setLevel('ERROR')

import numpy as np
from experionml import ExperionMLRegressor, ExperionMLModel
from sklearn.datasets import make_regression

from scikeras.wrappers import KerasRegressor
from keras.models import Sequential
from keras.layers import Dense

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Create data
X, y = make_regression(n_samples=1000, n_features=10, n_informative=5, n_targets=3)

In [ ]:
# Crie a rede neural
class NeuralNetwork(KerasRegressor):
    """Multioutput multilayer perceptron."""

    def __repr__(self):
        return "NeuralNetwork()"

    @staticmethod
    def _keras_build_fn(n_inputs, n_outputs, **kwargs):
        """Create the model's architecture."""
        model = Sequential()
        model.add(Dense(20, input_dim=n_inputs, activation="relu"))
        model.add(Dense(20, activation="relu"))
        model.add(Dense(n_outputs))
        model.compile(loss="mse", optimizer="adam")
        return model

In [ ]:
# Converta o modelo em um modelo do ExperionML
model = ExperionMLModel(
    estimator=NeuralNetwork(n_inputs=5, n_outputs=y.shape[1], epochs=100, verbose=0),
    name="NN",
    needs_scaling=True,  # Applies automated feature scaling before fitting
    native_multioutput=True,  # Não use um wrapper meta-estimador multioutput
)

## Executar o pipeline

In [ ]:
experionml = ExperionMLRegressor(X, y=y, verbose=2, random_state=1)

In [ ]:
# Mostre os modelos que oferecem suporte nativo a tarefas multioutput
experionml.available_models(native_multioutput=True)

In [ ]:
# Observe que adicionamos apenas 5 variáveis informativas ao conjunto de dados; vamos remover o restante
# Se usarmos como solver um modelo sem suporte nativo a multioutput, especifique o
# parâmetro importance_getter do rfe e retorne a média dos coeficientes ao longo das
# target columns
experionml.feature_selection(
    strategy="rfe",
    solver="ols",  # Isso se torna MultiOutputRegressor(OLS)
    n_features=5,
    importance_getter=lambda x: np.mean([e.coef_ for e in x.estimators_], axis=0),
)

In [ ]:
# Vamos treinar um modelo nativo, um não nativo e o nosso modelo personalizado
experionml.run(models=["Lasso", "RF", model], metric="mse", errors="raise")

In [ ]:
# E verificar quais modelos usaram um wrapper meta-estimador
for m in experionml.models:
    print(f"Estimator for {m} is: {experionml[m].estimator}")

## Analisar os resultados

In [ ]:
# Use o parâmetro target nos gráficos para especificar qual coluna alvo usar
experionml.plot_residuals(target=2)

In [ ]:
with experionml.canvas(3, 1, figsize=(900, 1300)):
    experionml.plot_errors(target=0)
    experionml.plot_errors(target=1)
    experionml.plot_errors(target=2)